In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split


df =  pd.read_json('../../../data/out/distillation/gsm8k_distilled_cot.jsonl', lines=True)

df

,question,answer,reasoning,distill_ans_correct,distill_response,distill_answer
0,Natalia sold clips to 48 of her friends in Apr...,72,Natalia sold 48/2 = <<48/2=24>>24 clips in May...,True,"To solve this problem, I will first identify t...",72
1,Weng earns $12 an hour for babysitting. Yester...,10,Weng earns 12/60 = $<<12/60=0.2>>0.2 per minut...,True,"To solve this problem, we need to first unders...",10
2,Betty is saving money for a new wallet which c...,5,"In the beginning, Betty has only 100 / 2 = $<<...",True,"To find out how much more money Betty needs, w...",5
3,"Julie is reading a 120-page book. Yesterday, s...",42,Maila read 12 x 2 = <<12*2=24>>24 pages today....,True,"To solve this problem, let's break it down ste...",42
4,James writes a 3-page letter to 2 different fr...,624,He writes each friend 3*2=<<3*2=6>>6 pages a w...,True,"To solve this problem, let's break it down int...",624
...,...,...,...,...,...,...
7468,"Very early this morning, Elise left home in a ...",5,"For the distance she traveled, Elise paid 23 -...",True,"To solve this problem, let's break it down int...",5
7469,Josh is saving up for a box of cookies. To rai...,3,He makes $.5 profit on each bracelet because 1...,True,"To solve this problem, let's break it down ste...",3
7470,Colin can skip at six times the speed that Bra...,4,Tony can skip at twice the speed that Bruce ca...,True,"## Step 1\nFirst, let's understand the problem...",4
7471,"Janet, a third grade teacher, is picking up th...",308,Janet needs 35 lunches for the kids + 5 for th...,True,"To find the total cost of the sack lunches, we...",308


In [2]:
df.value_counts('distill_answer')

distill_answer
4         207
10        201
6         200
5         198
20        176
         ... 
2802        1
28125       1
282         1
283         1
999000      1
Name: count, Length: 878, dtype: int64

In [3]:
df[df['distill_answer'] == '']

,question,answer,reasoning,distill_ans_correct,distill_response,distill_answer


In [4]:
df_entropy = pd.read_json('../../../data/out/single_token_entropy/gsm8k_phi4mini_single_token.jsonl', lines=True)
df_entropy

,question,answer,reasoning,reasoning_w_tools,entropy_ans_correct,entropy_value,entropy_ans
0,Natalia sold clips to 48 of her friends in Apr...,72,Natalia sold 48/2 = 24 clips in May.\nNatalia ...,Natalia sold 48/2 = <<48/2=24>>24 clips in May...,True,0.000660,72
1,Weng earns $12 an hour for babysitting. Yester...,10,Weng earns 12/60 = $0.2 per minute.\nWorking 5...,Weng earns 12/60 = $<<12/60=0.2>>0.2 per minut...,True,0.345512,10
2,Betty is saving money for a new wallet which c...,5,"In the beginning, Betty has only 100 / 2 = $50...","In the beginning, Betty has only 100 / 2 = $<<...",False,1.896246,10
3,"Julie is reading a 120-page book. Yesterday, s...",42,Maila read 12 x 2 = 24 pages today.\nSo she wa...,Maila read 12 x 2 = <<12*2=24>>24 pages today....,False,2.279938,36
4,James writes a 3-page letter to 2 different fr...,624,He writes each friend 3*2=6 pages a week\nSo h...,He writes each friend 3*2=<<3*2=6>>6 pages a w...,False,0.315048,72
...,...,...,...,...,...,...,...
7468,"Very early this morning, Elise left home in a ...",5,"For the distance she traveled, Elise paid 23 -...","For the distance she traveled, Elise paid 23 -...",True,0.001946,5
7469,Josh is saving up for a box of cookies. To rai...,3,He makes $.5 profit on each bracelet because 1...,He makes $.5 profit on each bracelet because 1...,False,0.966311,6
7470,Colin can skip at six times the speed that Bra...,4,Tony can skip at twice the speed that Bruce ca...,Tony can skip at twice the speed that Bruce ca...,False,0.009028,12
7471,"Janet, a third grade teacher, is picking up th...",308,Janet needs 35 lunches for the kids + 5 for th...,Janet needs 35 lunches for the kids + 5 for th...,False,0.648064,315


In [ ]:
# Merge entropy columns into df on 'question'
distill_cols = ['question', 'distill_ans_correct', 'distill_response', 'distill_answer']
df_entropy = df_entropy.merge(df[distill_cols], on='question', how='inner')

AttributeError: 'list' object has no attribute 'head'

In [10]:
df_entropy.head()

,question,answer,reasoning,reasoning_w_tools,entropy_ans_correct,entropy_value,entropy_ans,distill_ans_correct,distill_response,distill_answer
0,Natalia sold clips to 48 of her friends in Apr...,72,Natalia sold 48/2 = 24 clips in May.\nNatalia ...,Natalia sold 48/2 = <<48/2=24>>24 clips in May...,True,0.000660,72,True,"To solve this problem, I will first identify t...",72
1,Weng earns $12 an hour for babysitting. Yester...,10,Weng earns 12/60 = $0.2 per minute.\nWorking 5...,Weng earns 12/60 = $<<12/60=0.2>>0.2 per minut...,True,0.345512,10,True,"To solve this problem, we need to first unders...",10
2,Betty is saving money for a new wallet which c...,5,"In the beginning, Betty has only 100 / 2 = $50...","In the beginning, Betty has only 100 / 2 = $<<...",False,1.896246,10,True,"To find out how much more money Betty needs, w...",5
3,"Julie is reading a 120-page book. Yesterday, s...",42,Maila read 12 x 2 = 24 pages today.\nSo she wa...,Maila read 12 x 2 = <<12*2=24>>24 pages today....,False,2.279938,36,True,"To solve this problem, let's break it down ste...",42
4,James writes a 3-page letter to 2 different fr...,624,He writes each friend 3*2=6 pages a week\nSo h...,He writes each friend 3*2=<<3*2=6>>6 pages a w...,False,0.315048,72,True,"To solve this problem, let's break it down int...",624


In [11]:
df = df_entropy

In [12]:
df['norm_entropy'] = df['entropy_value'] / df['entropy_value'].max()

In [13]:
df['norm_entropy'].describe().round(5)

count    7473.00000
mean        0.23538
std         0.19912
min         0.00000
25%         0.06417
50%         0.20545
75%         0.35774
max         1.00000
Name: norm_entropy, dtype: float64

In [ ]:

train_valid_df = df.sort_values(by="norm_entropy", ascending=True)
N = len(train_valid_df)
print(f"Всего обучающих+валидационных примеров: {N}")

Всего обучающих+валидационных примеров: 7473


In [15]:
easy_df = train_valid_df.iloc[: int(0.33 * N)]
medium_df = train_valid_df.iloc[int(0.33 * N): int(0.66 * N)]
hard_df = train_valid_df.iloc[int(0.66 * N):]

In [19]:
hard_df.tail()

,question,answer,reasoning,reasoning_w_tools,entropy_ans_correct,entropy_value,entropy_ans,distill_ans_correct,distill_response,distill_answer,norm_entropy
4528,The treasurer of a football team must buy equi...,752,Full Equipment Price for one player: $25 + $15...,Full Equipment Price for one player: $25 + $15...,False,6.281243,1032.00,True,To find the total cost of the equipment for al...,752,0.957479
6181,Antonia is trying to improve her health by buy...,350,Antonia buys 5 different supplements where 3 b...,Antonia buys 5 different supplements where 3 b...,False,6.360667,710,True,"To solve this problem, let's first figure out ...",350,0.969586
5161,"Smaug the dragon hoards 100 gold coins, 60 sil...",2913,First figure out how many silver coins the 100...,First figure out how many silver coins the 100...,False,6.374475,3969,True,"To solve this problem, we need to convert the ...",2913,0.971691
2365,An archaeologist discovered three dig sites fr...,852,The third dig site was dated from the year 840...,The third dig site was dated from the year 840...,False,6.454645,6868 BC,True,"To solve this problem, let's analyze the infor...",852,0.983911
1202,A man is trying to maximize the amount of mone...,76,The mileage cost for the first apartment will ...,The mileage cost for the first apartment will ...,False,6.560189,-68,True,## Step 1: Calculate the total monthly cost fo...,76,1.000000


In [20]:
easy_df.to_json('../../../data/data_splits/entropy/phi/gsm8k_easy.jsonl', lines=True, orient='records')
medium_df.to_json('../../../data/data_splits/entropy/phi/gsm8k_medium.jsonl', lines=True, orient='records')
hard_df.to_json('../../../data/data_splits/entropy/phi/gsm8k_hard.jsonl', lines=True, orient='records')